<a href="https://colab.research.google.com/github/gcoelho3008/reconhecimento_facial/blob/main/reconhecimento_facial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Visão computacional com opnnCV

### **OpenCv**

In [ ]:
!pip install opencv-contrib-python

In [ ]:
from PIL import Image
from google.colab.patches import cv2_imshow
from google.colab import patches
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score

import seaborn
import os
import cv2
import numpy as np
import zipfile

In [ ]:
path = '/content/drive/MyDrive/Visão Computacional Guia Completo/Visão Computacional Guia Completo/Datasets/yalefaces.zip'
zip_object = zipfile.ZipFile(file=path, mode = 'r')
zip_object.extractall('./')
zip_object.close()

In [ ]:
print(os.listdir('/content/yalefaces'))

In [ ]:
def get_image_data():
  paths = [os.path.join('/content/yalefaces/train', f) for f in os.listdir('/content/yalefaces/train')]
  faces = []
  ids = []
  for path in paths:
    imagem = Image.open(path).convert('L')
    imagem_np = np.array(imagem, 'uint8')
    id = int(os.path.split(path)[1].split('.')[0].replace('subject', ''))
    ids.append(id)
    faces.append(imagem_np)

  return np.array(ids), faces

In [ ]:
ids, faces = get_image_data()

In [ ]:
ids

In [ ]:
len(ids)

In [ ]:
faces[0], faces[0].shape

### Treinamento do classificador LBPH

In [ ]:
8 * 8,9 * 9

In [ ]:
# threshold: 1.7976931348623157e+38
# radius: 1
# neighbors: 8
# grid_x: 8
# grid_y: 8

lbph_classifier = cv2.face.LBPHFaceRecognizer_create(radius=4, neighbors=14, grid_x=9, grid_y=9)
lbph_classifier.train(faces, ids)
lbph_classifier.write('lbph_classifier.yml')

### Reconhecimento de Faces

In [ ]:
lbph_face_classifier = cv2.face.LBPHFaceRecognizer_create()
lbph_face_classifier.read('lbph_classifier.yml')

In [ ]:
imagem_teste = '/content/yalefaces/test/subject10.sad.gif'

In [ ]:
imagem = Image.open(imagem_teste).convert('L')
imagem_np = np.array(imagem, 'uint8')
imagem_np

In [ ]:
imagem_np.shape

In [ ]:
previsao = lbph_face_classifier.predict(imagem_np)
previsao

In [ ]:
previsao[0]

In [ ]:
saida_esperada = int(os.path.split(imagem_teste)[1].split('.')[0].replace('subject', ''))
saida_esperada

In [ ]:
cv2.putText(imagem_np, 'Pred: ' + str(previsao[0]), (10, 30), cv2.FONT_HERSHEY_COMPLEX_SMALL, 1, (0, 255, 0))
cv2.putText(imagem_np, 'Expr: ' + str(saida_esperada), (10,50), cv2.FONT_HERSHEY_COMPLEX_SMALL, 1, (0, 255, 0))
cv2_imshow(imagem_np)


### Avaliação do Classificador

In [ ]:
paths = [os.path.join('/content/yalefaces/test', f) for f in os.listdir('/content/yalefaces/test')]
previsoes = []
saidas_esperadas = []
for path in paths:
 # print(path)
  imagem = Image.open(path).convert('L')
  imagem_np = np.array(imagem, 'uint8')
  previsao, _ = lbph_face_classifier.predict(imagem_np)
 # print(previsao)
  saida_esperada = int(os.path.split(path)[1].split('.')[0].replace('subject', ''))
  #print(saida_esperada)

  previsoes.append(previsao)
  saidas_esperadas.append(saida_esperada)

In [ ]:
type(previsoes), type(saidas_esperadas)

In [ ]:
previsoes = np.array(previsoes)
saidas_esperadas = np.array(saidas_esperadas)

In [ ]:
type(previsoes), type(saidas_esperadas)


In [ ]:
previsoes

In [ ]:
saidas_esperadas

In [ ]:
accuracy_score(saidas_esperadas, previsoes)

In [ ]:
len(previsoes)

In [ ]:
cm = confusion_matrix(saidas_esperadas, previsoes)
cm

In [ ]:
seaborn.heatmap(cm, annot=True);

In [ ]:
acc = accuracy_score(saidas_esperadas, previsoes)
print(f'Acurácia: {acc*100:.2f}%')

### Detecção de pontos faciais com dlib

In [ ]:
import dlib

In [ ]:
detector_face = dlib.get_frontal_face_detector()
detector_pontos = dlib.shape_predictor('/content/drive/MyDrive/Visão Computacional Guia Completo/Visão Computacional Guia Completo/Weights/shape_predictor_68_face_landmarks.dat')


In [ ]:
imagem = cv2.imread('/content/drive/MyDrive/Visão Computacional Guia Completo/Visão Computacional Guia Completo/Images/people2.jpg')
deteccoes = detector_face(imagem, 1)
for face in deteccoes:
  pontos = detector_pontos(imagem, face)
  for ponto in pontos.parts():
    cv2.circle(imagem, (ponto.x, ponto.y), 2, (0, 255, 0), 1)
  #print(pontos.parts())
  #print(len(pontos.parts()))

  l, t, r, b = face.left(), face.top(), face.right(), face.bottom()
  cv2.rectangle(imagem, (l, t), (r, b), (0, 255, 0), 2)
cv2_imshow(imagem)
